# Querying and Scanning

## Overview
DynamoDB provides two read operations for fetching multiple items: **Query** and **Scan**.

**The golden rule: always Query, never Scan in production.**

| | Query | Scan |
|---|---|---|
| How it works | Reads items matching a PK (and optionally SK range) | Reads every item in the table or index |
| Requires | At minimum the partition key | Nothing (reads everything) |
| Read cost | Proportional to result set | Proportional to **entire table** |
| Use when | You know the partition key | Data migration, small tables, one-off exports |

**Key condition expressions** (used only in Query):
```python
Key("site_id").eq("SITE#001")                         # exact PK match
Key("obs_sk").begins_with("OBS#2023")                 # SK prefix
Key("obs_sk").between("OBS#2023-04-01", "OBS#2023-06-30#zzz")  # SK range
Key("obs_sk").gt("OBS#2023-06-01")                    # SK comparison
```

**Filter expressions** (used in both Query and Scan — but do NOT reduce read costs):
```python
Attr("at_risk").eq(True)
Attr("count").gte(5)
Attr("method").is_in(["Electrofishing", "Visual"])
Attr("notes").exists()
```

---

In [1]:

print("=== Query vs Scan: the most important performance distinction ===")
comparison = [
    ("Query",  "Uses primary key or index", "O(result set size)",  "Always preferred; scales well"),
    ("Scan",   "Reads entire table",        "O(table size)",       "Avoid on large tables; use sparingly"),
]
print(f"  {'Operation':<10s}  {'How it works':<32s}  {'Cost':<22s}  Guidance")
print("  " + "-"*80)
for op, how, cost, guidance in comparison:
    print(f"  {op:<10s}  {how:<32s}  {cost:<22s}  {guidance}")

print()
print("Query requires at minimum the partition key.")
print("Scan reads every item; FilterExpression reduces the RESPONSE, not the READ cost.")
print()
print("Query examples (ecological observations table):")
examples = [
    ("All observations at site 001",
     "Query: PK='SITE#001'"),
    ("Observations at site 001 in 2023",
     "Query: PK='SITE#001', SK begins_with 'OBS#2023'"),
    ("Observations at site 001 between two dates",
     "Query: PK='SITE#001', SK between 'OBS#2023-04-01' and 'OBS#2023-06-30'"),
    ("A specific observation",
     "GetItem: PK='SITE#001', SK='OBS#2023-04-10#001'"),
    ("All observations for species SP#SALMON across all sites",
     "Requires a GSI with species_id as PK (see indexes notebook)"),
]
for q, method in examples:
    print(f"  Access: {q}")
    print(f"  Method: {method}")
    print()


=== Query vs Scan: the most important performance distinction ===
  Operation   How it works                      Cost                    Guidance
  --------------------------------------------------------------------------------
  Query       Uses primary key or index         O(result set size)      Always preferred; scales well
  Scan        Reads entire table                O(table size)           Avoid on large tables; use sparingly

Query requires at minimum the partition key.
Scan reads every item; FilterExpression reduces the RESPONSE, not the READ cost.

Query examples (ecological observations table):
  Access: All observations at site 001
  Method: Query: PK='SITE#001'

  Access: Observations at site 001 in 2023
  Method: Query: PK='SITE#001', SK begins_with 'OBS#2023'

  Access: Observations at site 001 between two dates
  Method: Query: PK='SITE#001', SK between 'OBS#2023-04-01' and 'OBS#2023-06-30'

  Access: A specific observation
  Method: GetItem: PK='SITE#001', SK='OBS#

---
## Query with KeyConditionExpression

In [2]:

print("=== Query with KeyConditionExpression ===")
query_code = '''
from boto3.dynamodb.conditions import Key, Attr

table = dynamodb.Table("EcoObservations")

# ── All items for a site (PK only) ──────────────────────────────
response = table.query(
    KeyConditionExpression=Key("site_id").eq("SITE#001")
)
items = response["Items"]

# ── Observations for a site in 2023 (PK + SK prefix) ────────────
response = table.query(
    KeyConditionExpression=(
        Key("site_id").eq("SITE#001") &
        Key("obs_sk").begins_with("OBS#2023")
    )
)

# ── Observations in a date range ─────────────────────────────────
response = table.query(
    KeyConditionExpression=(
        Key("site_id").eq("SITE#001") &
        Key("obs_sk").between("OBS#2023-04-01", "OBS#2023-06-30#zzz")
    ),
    # FilterExpression is applied AFTER reading; does not reduce RCU cost
    FilterExpression=Attr("at_risk").eq(True),
    # Projection: only return these attributes
    ProjectionExpression="obs_sk, common_name, #cnt, at_risk",
    ExpressionAttributeNames={"#cnt": "count"},
    # Sort order (default is ascending by SK)
    ScanIndexForward=False,        # False = descending (most recent first)
    Limit=20,                      # max items per page
)

# ── Pagination: handle LastEvaluatedKey ──────────────────────────
all_items = []
params = {"KeyConditionExpression": Key("site_id").eq("SITE#001")}
while True:
    response = table.query(**params)
    all_items.extend(response["Items"])
    last_key = response.get("LastEvaluatedKey")
    if not last_key:
        break
    params["ExclusiveStartKey"] = last_key
print(f"Total items fetched: {len(all_items)}")
'''
print(query_code)


=== Query with KeyConditionExpression ===

from boto3.dynamodb.conditions import Key, Attr

table = dynamodb.Table("EcoObservations")

# ── All items for a site (PK only) ──────────────────────────────
response = table.query(
    KeyConditionExpression=Key("site_id").eq("SITE#001")
)
items = response["Items"]

# ── Observations for a site in 2023 (PK + SK prefix) ────────────
response = table.query(
    KeyConditionExpression=(
        Key("site_id").eq("SITE#001") &
        Key("obs_sk").begins_with("OBS#2023")
    )
)

# ── Observations in a date range ─────────────────────────────────
response = table.query(
    KeyConditionExpression=(
        Key("site_id").eq("SITE#001") &
        Key("obs_sk").between("OBS#2023-04-01", "OBS#2023-06-30#zzz")
    ),
    # FilterExpression is applied AFTER reading; does not reduce RCU cost
    FilterExpression=Attr("at_risk").eq(True),
    # Projection: only return these attributes
    ProjectionExpression="obs_sk, common_name, #cnt, at_risk",
    

---
## Scan: when and how

In [3]:

print("=== Scan: when and how ===")
scan_code = '''
# Scan reads every item in the table (or every item in an index).
# Use ONLY when:
#   - The table is small (< a few thousand items)
#   - You need a one-off data export or migration
#   - No suitable index exists and you accept the full-table read cost

# ── Basic scan with filter ───────────────────────────────────────
response = table.scan(
    FilterExpression=Attr("region").eq("Atlantic") & Attr("active").eq(True),
    # FilterExpression does NOT reduce read cost — all items are read;
    # non-matching items are discarded BEFORE being returned to you
)
items = response["Items"]
consumed = response["ConsumedCapacity"]   # how many RCUs were used

# ── Parallel scan: divide table into N segments ──────────────────
# For large tables, scan in parallel from multiple threads/processes
import threading

def scan_segment(segment, total_segments, results):
    response = table.scan(
        Segment=segment,
        TotalSegments=total_segments,
    )
    results.extend(response["Items"])

results = []
threads = []
for seg in range(4):   # 4 parallel workers
    t = threading.Thread(target=scan_segment, args=(seg, 4, results))
    threads.append(t)
    t.start()
for t in threads:
    t.join()
print(f"Parallel scan collected {len(results)} items")

# ── Scan for data export ─────────────────────────────────────────
# Better alternative: enable DynamoDB export to S3 (point-in-time)
# and query with Athena — no read capacity consumed at all
'''
print(scan_code)

print("Scan cost vs Query cost:")
print("  A 1,000-item table fully scanned: 1,000 item reads")
print("  A 1,000-item table queried by PK: N item reads (N = matching items)")
print("  For a table with 100M items, a Scan costs 100M reads regardless of filter selectivity.")
print("  Always prefer Query. If Query is impossible, redesign your keys or add a GSI.")


=== Scan: when and how ===

# Scan reads every item in the table (or every item in an index).
# Use ONLY when:
#   - The table is small (< a few thousand items)
#   - You need a one-off data export or migration
#   - No suitable index exists and you accept the full-table read cost

# ── Basic scan with filter ───────────────────────────────────────
response = table.scan(
    FilterExpression=Attr("region").eq("Atlantic") & Attr("active").eq(True),
    # FilterExpression does NOT reduce read cost — all items are read;
    # non-matching items are discarded BEFORE being returned to you
)
items = response["Items"]
consumed = response["ConsumedCapacity"]   # how many RCUs were used

# ── Parallel scan: divide table into N segments ──────────────────
# For large tables, scan in parallel from multiple threads/processes
import threading

def scan_segment(segment, total_segments, results):
    response = table.scan(
        Segment=segment,
        TotalSegments=total_segments,
    )
    resul

---
## Common Pitfalls

**1. Applying FilterExpression and expecting it to reduce read costs**
FilterExpression runs AFTER DynamoDB reads items from the table or index. A Scan with `FilterExpression=Attr('at_risk').eq(True)` still reads every item and consumes capacity for all of them -- only the returned payload is filtered. To reduce actual read costs, the filter condition must be the partition key (for Query) or a key on an index (GSI/LSI).

**2. Forgetting to paginate responses**
DynamoDB returns at most 1 MB of data per request (or the number set by `Limit`). If there are more results, `LastEvaluatedKey` is set in the response. Ignoring it means you silently receive only the first page. Always loop on `LastEvaluatedKey` until it is absent.

**3. ScanIndexForward misunderstanding**
`ScanIndexForward=False` sorts results descending by sort key -- it does NOT mean a backward Scan. It only applies to Query operations. For a Scan, items are returned in undefined order regardless of `ScanIndexForward`.

**4. Using Scan in Lambda functions triggered by user requests**
A Lambda triggered per API request that does a Scan on a growing table is a time bomb. At 100K items the Scan takes seconds and costs significantly; at 10M items it times out. Any Scan in a user-facing code path must be replaced with a Query against a properly designed index.

**5. `begins_with` on sort key not working as expected for dates**
`Key('obs_sk').begins_with('OBS#2023')` works correctly when sort keys are formatted as `OBS#YYYY-MM-DD#ID` because the prefix is consistently `OBS#2023`. But `begins_with('OBS#')` returns ALL observations regardless of date -- which may be intentional or not. Always test prefix queries against the exact format of your sort key values.

**6. Not setting ConsistentRead=True when strongly consistent reads are required**
DynamoDB reads are eventually consistent by default. A GetItem immediately after a PutItem may return stale data. For scenarios requiring read-your-own-writes consistency (e.g. verifying a write succeeded), set `ConsistentRead=True`. This doubles the RCU cost but guarantees the latest value.


---
*sql_methods_library - Samantha McGarrigle*